# 06 — Platform KPIs (in-notebook Executive Overview)

Renders the same KPI set as the Power BI "Executive Overview" page (`ARCHITECTURE.md §13`) directly in the notebook with matplotlib — for anyone reviewing this project without Power BI access. Metric definitions follow `docs/semantic-dictionary.md` (Revenue = completed orders only, excluding canceled — see `ARCHITECTURE.md §11`).

Run `make seed` from the repo root before running this notebook. Self-contained — does not import the other notebooks.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OLIST_DIR = ROOT / "data" / "raw" / "olist"

required = [
    OLIST_DIR / "olist_orders_dataset.csv",
    OLIST_DIR / "olist_order_payments_dataset.csv",
    OLIST_DIR / "olist_customers_dataset.csv",
]
missing = [str(p) for p in required if not p.exists()]
if missing:
    raise FileNotFoundError("Missing: " + ", ".join(missing) + " — run `make download-olist` / `make seed` first.")

orders = pd.read_csv(OLIST_DIR / "olist_orders_dataset.csv", parse_dates=["order_purchase_timestamp"])
payments = pd.read_csv(OLIST_DIR / "olist_order_payments_dataset.csv")
customers = pd.read_csv(OLIST_DIR / "olist_customers_dataset.csv")
print(f"orders: {len(orders):,} | payments: {len(payments):,} | customers: {len(customers):,}")

## Compute headline KPIs

`Revenue`, `Orders`, `Customers`, `AOV` follow the canonical definitions in `docs/semantic-dictionary.md`: completed orders only (`order_status` not in `canceled`/`unavailable`).

In [ ]:
EXCLUDED_STATUSES = {"canceled", "unavailable"}
completed = orders[~orders["order_status"].isin(EXCLUDED_STATUSES)]

order_revenue = (
    payments.groupby("order_id")["payment_value"].sum()
    .reindex(completed["order_id"]).fillna(0)
)

revenue = order_revenue.sum()
n_orders = len(completed)
n_customers = customers["customer_unique_id"].nunique()
aov = revenue / n_orders if n_orders else 0
repeat_customers = (
    orders.merge(customers[["customer_id", "customer_unique_id"]], on="customer_id")
    .groupby("customer_unique_id")["order_id"].nunique()
)
repeat_rate = (repeat_customers > 1).mean()

kpis = {
    "Revenue": f"R$ {revenue:,.0f}",
    "Orders": f"{n_orders:,}",
    "Customers": f"{n_customers:,}",
    "AOV": f"R$ {aov:,.2f}",
    "Repeat Rate": f"{repeat_rate:.1%}",
}
kpis

## KPI cards

In [ ]:
fig, axes = plt.subplots(1, len(kpis), figsize=(3 * len(kpis), 2.2))
for ax, (label, value) in zip(axes, kpis.items()):
    ax.axis("off")
    ax.add_patch(plt.Rectangle((0, 0), 1, 1, transform=ax.transAxes,
                                facecolor="#F2F2F2", edgecolor="#CCCCCC"))
    ax.text(0.5, 0.62, value, ha="center", va="center", fontsize=16, fontweight="bold")
    ax.text(0.5, 0.22, label, ha="center", va="center", fontsize=10, color="#555555")
plt.suptitle("Executive Overview", fontsize=13, y=1.05)
plt.tight_layout()
plt.show()

## Revenue trend and top states

In [ ]:
completed_with_revenue = completed.assign(revenue=order_revenue.values)
monthly_revenue = (
    completed_with_revenue.set_index("order_purchase_timestamp")["revenue"]
    .resample("MS").sum()
)

orders_by_state = (
    orders.merge(customers[["customer_id", "customer_state"]], on="customer_id")
    ["customer_state"].value_counts().head(10)
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
monthly_revenue.plot(ax=axes[0], color="#4C72B0")
axes[0].set_title("Revenue over time")
axes[0].set_ylabel("R$")

orders_by_state.plot.bar(ax=axes[1], color="#55A868")
axes[1].set_title("Orders by state (top 10)")
axes[1].set_ylabel("orders")

plt.tight_layout()
plt.show()

## Data Quality Score (compact)

A one-line summary of notebook 01's fuller breakdown, included here so the Executive Overview always ships with a trust signal next to the business numbers — exactly the pairing `ARCHITECTURE.md §7` argues for ("the platform reports its own trustworthiness, not just business metrics").

In [ ]:
key_columns = {
    "orders.order_id": orders["order_id"],
    "orders.customer_id": orders["customer_id"],
    "customers.customer_unique_id": customers["customer_unique_id"],
}
completeness_scores = [1 - s.isna().mean() for s in key_columns.values()]
uniqueness_score = 1 - orders["order_id"].duplicated().mean()
dq_score = (sum(completeness_scores) / len(completeness_scores) + uniqueness_score) / 2

print(f"Data Quality Score (compact): {dq_score:.1%}")
print("See 01_data_quality_overview.ipynb for the full breakdown by dimension.")

## Maps to the real pipeline

- Power BI "Executive Overview" page (`powerbi/`) — the production version of this page, sourced from the semantic layer (Snowflake Semantic Views / Unity Catalog Metric Views), never from raw tables directly (ADR-005) — this notebook computes straight from Olist CSVs only because it has no warehouse to query locally.
- `docs/semantic-dictionary.md` — the canonical Revenue/AOV/Repeat Rate definitions this notebook implements a local copy of.
- `01_data_quality_overview.ipynb`, `03_churn_model_explainability.ipynb`, `04_customer_segmentation.ipynb` — the fuller versions of the DQ score and (not shown here, since it needs the synthetic CRM join) churn/segment KPIs.